In [1]:
import skimage as ski
import numpy as np
import napari
import glob
import scipy.ndimage as ndi
import tifffile
import cv2
import plotly.express as px


In [2]:
viewer = napari.Viewer()

In [24]:
fnames = glob.glob('../20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2/projected_good/*_01.tif')
fnames.sort()
fnames

['../20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2/projected_good\\A01_01.tif',
 '../20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2/projected_good\\A02_01.tif',
 '../20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2/projected_good\\A03_01.tif',
 '../20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2/projected_good\\A04_01.tif',
 '../20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2/projected_good\\A05_01.tif',
 '../20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2/projected_good\\A06_01.tif',
 '../20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2/projected_good\\A07_01.tif',
 '../20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2/projected_good\\A08_01.tif',
 '../20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2/projected_good\\A09_01.tif',
 '../20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2/projected_good\\A10_01.tif',
 '../20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2/projec

In [73]:
def get_all_tiles(fname):
    ids = ['01', '02', '03', '04', '05', '06', '07', '08', '09']
    tiles = []
    for id in ids:
        cname = (fname.replace('_01.tif', f'_{id}.tif'))
        img = ski.io.imread(cname)
        tiles.append(img)
    tiles = np.array(tiles)
    if len(tiles.shape)==4:
        tiles = tiles[:,np.newaxis,:,:,:]
    return tiles

In [74]:
def montage_fname(fname):
    tiles = get_all_tiles(fname)

    c0_median = ski.io.imread('ch1_median.tif')
    c1_median = ski.io.imread('ch2_median.tif')

    tiles[:,:,:,:,0] = np.clip(tiles[:,:, :, :, 0].astype(np.float32) - c0_median, 0, None).astype(np.uint16)
    tiles[:,:,:,:,1] = np.clip(tiles[:,:, :, :, 1].astype(np.float32) - c1_median, 0, None).astype(np.uint16)

    tiles = np.moveaxis(tiles, 1, 3)
    tiles = np.reshape(tiles, (tiles.shape[0], tiles.shape[1], tiles.shape[2], tiles.shape[3]*tiles.shape[4]))

    reorder = [1, 2, 3, 5, 0, 4, 6, 7, 8]
    tiles = tiles[reorder]

    montage = ski.util.montage(tiles, channel_axis=-1, grid_shape=(3, 3),)
    montage = np.reshape(montage, (montage.shape[0], montage.shape[1], montage.shape[2]//3, 3))
    montage = np.moveaxis(montage, 2, 0)
    return montage

# Batch Montage All

In [75]:
dirs = glob.glob('../2026*/')
dirs.sort()
dirs

['..\\20260818_HP_DrugScreen1__2026-08-18T12_51_56-Measurement 2\\',
 '..\\20260819_HP_DrugScreen2__2026-08-19T07_31_31-Measurement 1\\',
 '..\\20260819_HP_DrugScreen3__2026-08-19T13_30_04-Measurement 2\\',
 '..\\20260820_HP_DrugScreen4__2026-08-20T13_56_08-Measurement 1\\',
 '..\\20260821_HP_DrugScreen5__2026-08-21T13_35_45-Measurement 1\\']

In [76]:
root_names = []
for fname in fnames:
    root_names.append(fname.split('\\')[-1])
root_names

['A01_01.tif',
 'A02_01.tif',
 'A03_01.tif',
 'A04_01.tif',
 'A05_01.tif',
 'A06_01.tif',
 'A07_01.tif',
 'A08_01.tif',
 'A09_01.tif',
 'A10_01.tif',
 'A11_01.tif',
 'A12_01.tif',
 'A13_01.tif',
 'A14_01.tif',
 'A15_01.tif',
 'A16_01.tif',
 'A17_01.tif',
 'A18_01.tif',
 'A19_01.tif',
 'A20_01.tif',
 'A21_01.tif',
 'A22_01.tif',
 'A23_01.tif',
 'A24_01.tif',
 'B01_01.tif',
 'B02_01.tif',
 'B03_01.tif',
 'B04_01.tif',
 'B05_01.tif',
 'B06_01.tif',
 'B07_01.tif',
 'B08_01.tif',
 'B09_01.tif',
 'B10_01.tif',
 'B11_01.tif',
 'B12_01.tif',
 'B13_01.tif',
 'B14_01.tif',
 'B15_01.tif',
 'B16_01.tif',
 'B17_01.tif',
 'B18_01.tif',
 'B19_01.tif',
 'B20_01.tif',
 'B21_01.tif',
 'B22_01.tif',
 'B23_01.tif',
 'B24_01.tif',
 'C01_01.tif',
 'C02_01.tif',
 'C03_01.tif',
 'C04_01.tif',
 'C05_01.tif',
 'C06_01.tif',
 'C07_01.tif',
 'C08_01.tif',
 'C09_01.tif',
 'C10_01.tif',
 'C11_01.tif',
 'C12_01.tif',
 'C13_01.tif',
 'C14_01.tif',
 'C15_01.tif',
 'C16_01.tif',
 'C17_01.tif',
 'C18_01.tif',
 'C19_01.t

In [77]:
for root_name in root_names:
    print(root_name)
    montages = []
    for dir in dirs:
        montage = montage_fname(f'{dir}/projected_good/{root_name}')
        montages.append(montage)
    montages = np.concatenate(montages, axis=0)
    ski.io.imsave('concatenated/' + root_name.replace('_01.tif', '_montage.tif'), montages)


A01_01.tif
A02_01.tif
A03_01.tif
A04_01.tif
A05_01.tif
A06_01.tif
A07_01.tif
A08_01.tif
A09_01.tif
A10_01.tif
A11_01.tif
A12_01.tif
A13_01.tif
A14_01.tif
A15_01.tif
A16_01.tif
A17_01.tif
A18_01.tif
A19_01.tif
A20_01.tif
A21_01.tif
A22_01.tif
A23_01.tif
A24_01.tif
B01_01.tif
B02_01.tif
B03_01.tif
B04_01.tif
B05_01.tif
B06_01.tif
B07_01.tif
B08_01.tif
B09_01.tif
B10_01.tif
B11_01.tif
B12_01.tif
B13_01.tif
B14_01.tif
B15_01.tif
B16_01.tif
B17_01.tif
B18_01.tif
B19_01.tif
B20_01.tif
B21_01.tif
B22_01.tif
B23_01.tif
B24_01.tif
C01_01.tif
C02_01.tif
C03_01.tif
C04_01.tif
C05_01.tif
C06_01.tif
C07_01.tif
C08_01.tif
C09_01.tif
C10_01.tif
C11_01.tif
C12_01.tif
C13_01.tif
C14_01.tif
C15_01.tif
C16_01.tif
C17_01.tif
C18_01.tif
C19_01.tif
C20_01.tif
C21_01.tif
C22_01.tif
C23_01.tif
C24_01.tif
D01_01.tif
D02_01.tif
D03_01.tif
D04_01.tif
D05_01.tif
D06_01.tif
D07_01.tif
D08_01.tif
D09_01.tif
D10_01.tif
D11_01.tif
D12_01.tif
D13_01.tif
D14_01.tif
D15_01.tif
D16_01.tif
D17_01.tif
D18_01.tif
D19_01.tif